ODL_lab_FNO_2026.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1Gur15kGt_sW27mVcoJ4RggSV0vOdL9Se

# TP — Fourier Neural Operator sur Darcy Flow

Ce notebook est inspiré du notebook *Training on Darcy Flow* du **bootcamp NeuralOperator** :
configuration -> data Darcy -> baseline (U‑Net) -> modèle principal (FNO/TFNO) -> entraînement avec `Trainer` -> test 32×32 / 64×64 -> ablations.


Si un GPU est disponible, vous pourrez simplement augmenter `n_train` et `n_epochs`.

---

In [ ]:
# --- Dépendances NeuralOperator ---
# Le package s'installe via "neuraloperator" mais s'importe via "neuralop".
# Sur Colab, exécutez cette cellule.

import sys, importlib, pkgutil

print("Python:", sys.version)
if sys.version_info < (3, 9):
    raise RuntimeError("NeuralOperator requiert Python >= 3.9 (cf. PyPI). Utilisez Colab ou mettez à jour votre environnement.")

if pkgutil.find_loader("neuralop") is None:
    print("Installation de neuraloperator (PyPI)...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "neuraloperator"], check=True)
    importlib.invalidate_caches()

import neuralop
print("neuralop import OK | version:", getattr(neuralop, "__version__", "unknown"))

# Installation (Colab/VM). Sur une machine déjà configurée, vous pouvez commenter.
# !pip -q install neuraloperator

import os
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.manual_seed(0)
np.random.seed(0)

## 0. Rappels : Darcy flow (milieux poreux)

On modélise un écoulement dans un **milieu poreux** (sol, roche, filtre).

- $u(x,y)$ : **pression** (ou charge hydraulique)
- $a(x,y)$ : **perméabilité / conductivité** (varie spatialement)
- $f(x,y)$ : terme source (injection/extraction), souvent fixé dans le benchmark

### Équations
**Loi de Darcy** (flux/vitesse) :
$$
\mathbf{v}(x,y) = -a(x,y)\,\nabla u(x,y)
$$

**Conservation de la masse** (incompressible) :
$$
\nabla \cdot \mathbf{v}(x,y) = f(x,y)
$$

Donc la PDE :
$$
-\nabla \cdot (a(x,y)\nabla u(x,y)) = f(x,y)
$$
avec typiquement **Dirichlet** $u=0$ sur le bord du carré.

### Ce qu’on apprend (apprentissage d’opérateur)
On apprend une application :
$$
\mathcal{G}: a(\cdot)\mapsto u(\cdot)
$$
c.-à-d. **champ -> champ**. C’est plus proche d’une “résolution de PDE par apprentissage” que d’une régression classique.

---

## 0bis. Rappel : qu’est-ce qu’un Fourier Neural Operator (FNO) ?

Un FNO apprend aussi une application **champ -> champ**, mais avec une idée clé :

> Au lieu d’appliquer uniquement des convolutions locales (CNN), on apprend des transformations dans le **domaine de Fourier**.

### Intuition (version simple)
Pour un tenseur $x(x,y)$ (plusieurs canaux) :
1. **FFT 2D** : $\hat x = \mathcal{F}(x)$
2. On garde seulement les **basses fréquences** (les “modes”) : on tronque à `n_modes=(m1,m2)`
3. On applique une transformation linéaire apprise sur ces modes (poids complexes) : $\hat y_{k} = W_k \hat x_{k}$
4. **iFFT** : $y = \mathcal{F}^{-1}(\hat y)$

Ensuite on ajoute une branche “pointwise” (1×1 conv / MLP par pixel) + non‑linéarités et on empile plusieurs blocs.

### Hyperparamètres clés
- `n_modes=(m1,m2)` : combien de fréquences basses on garde.
  - petit -> sortie trop lisse (perte de détails)
  - grand -> plus de détails, mais plus coûteux (et parfois plus instable)
- `hidden_channels` : largeur (nombre de canaux internes)
- `n_layers` : profondeur (nombre de blocs FNO)

### TFNO (Tensorized FNO)
`TFNO` = variante factorisée (Tucker, etc.) qui vise à **réduire le coût** quand on augmente la taille (modes / canaux).
Dans ce TP, on utilise TFNO si possible, sinon FNO.

### Pourquoi “généraliser en résolution” ?
Dans beaucoup de settings, on entraîne sur 32×32 et on teste sur 64×64 (“zero-shot super-resolution”).
L’idée est que le FNO apprend une transformation “spectrale” compatible avec des grilles plus fines.

---

## 1. Configuration (bootcamp-style)

Valeurs par défaut **CPU-friendly**.

Règle pratique :
- Si ça ne tourne pas sur CPU : baissez `n_train`, `n_epochs`, puis `hidden_channels`, puis `n_modes`.
- Si vous voulez utiliser un GPU (device="cuda"): augmentez d’abord `n_train`, puis `n_epochs`, puis `hidden_channels`.

Paramètres importants :
- `training_loss`: `"l2"` (amplitude) ou `"h1"` (amplitude + structure fine)
- `n_modes`: détail spectral capturable
- `hidden_channels`: capacité
- `n_layers`: profondeur

---

In [ ]:
config = {
    "data": {
        "n_train": 256,          # CPU: 128–512 ; GPU: 2000+
        "batch_size": 16,
        "n_tests": [64, 64],     # 32×32 et 64×64
        "test_resolutions": [32, 64],
        "test_batch_sizes": [16, 8],
        "train_resolution": 32,
    },
    "unet": {
        "base_channels": 16,     # CPU: 8–16 ; GPU: 32–64
        "depth": 4,
    },
    "fno": {
        "n_modes": (12, 12),     # ex. (8,8), (16,16), (24,24)
        "hidden_channels": 16,   # CPU: 16 ; GPU: 64+
        "n_layers": 4,
        "factorization": "tucker",   # mettre None pour FNO “plein”
        "implementation": "factorized",
        "rank": 0.2,
        "projection_channels": 128,
    },
    "opt": {
        "n_epochs": 10,          # CPU: 5–20 ; GPU: 30–80
        "learning_rate": 5e-3,
        "weight_decay": 1e-4,
        "scheduler": "CosineAnnealingLR",
        "scheduler_T_max": 50,
        "step_size": 20,
        "gamma": 0.5,
        "training_loss": "h1",   # "l2" ou "h1"
    }
}

## 2. Charger le dataset Darcy (NeuralOperator)

Le loader renvoie :
- `train_loader`
- `test_loaders` (souvent 32×32 et 64×64)
- `data_processor` (normalisation + encodages)

In [ ]:
# Compat: API récente vs plus ancienne
load_fn = None
load_api = None

try:
    from neuralop.data.datasets import load_darcy_flow_small as load_fn
    load_api = "load_darcy_flow_small"
except Exception:
    try:
        from neuralop.datasets import load_darcy_pt as load_fn
        load_api = "load_darcy_pt"
    except Exception as e:
        raise RuntimeError(
            "Impossible d'importer un loader Darcy depuis NeuralOperator. "
            "Vérifiez l'installation de neuraloperator."
        ) from e

print("Using loader:", load_api)

if load_api == "load_darcy_flow_small":
    train_loader, test_loaders, data_processor = load_fn(
        n_train=config["data"]["n_train"],
        batch_size=config["data"]["batch_size"],
        n_tests=config["data"]["n_tests"],
        test_resolutions=config["data"]["test_resolutions"],
        test_batch_sizes=config["data"]["test_batch_sizes"],
    )
else:
    # Ancienne API : nécessite des fichiers .pt locaux
    data_folder = "./data/darcy_flow"
    train_loader, test_loaders, data_processor = load_fn(
        data_folder,
        train_resolution=config["data"]["train_resolution"],
        n_train=config["data"]["n_train"],
        batch_size=config["data"]["batch_size"],
        test_resolutions=config["data"]["test_resolutions"],
        n_tests=config["data"]["n_tests"],
        test_batch_sizes=config["data"]["test_batch_sizes"],
        positional_encoding=True,
        encode_input=True,
        encode_output=False,
    )

sample = next(iter(train_loader))
if isinstance(sample, (list, tuple)):
    x, y = sample
else:
    x, y = sample["x"], sample["y"]

print("Batch x:", x.shape, "Batch y:", y.shape)
in_channels = x.shape[1]
print("Inferred in_channels:", in_channels)

## 3. Visualisations : champs et flux

On visualise $a(x,y)$, $u(x,y)$, et le flux $\mathbf{v}=-a \nabla u$.

In [ ]:
def show_darcy_example(x, y, title=""):
    x0 = x[3].detach().cpu()
    u0 = y[3, 0].detach().cpu()

    a = x0[0]  # hypothèse : canal 0 = perméabilité

    # Vérifier bord (souvent u≈0)
    border = torch.cat([u0[0, :], u0[-1, :], u0[:, 0], u0[:, -1]])
    print("u border mean| |:", border.abs().mean().item(), " max|u|:", border.abs().max().item())

    fig, axs = plt.subplots(1, 3, figsize=(13, 3.5))
    axs[0].set_title("a(x,y) (perméabilité)")
    im0 = axs[0].imshow(a, origin="lower"); plt.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)

    axs[1].set_title("u(x,y) (pression)")
    im1 = axs[1].imshow(u0, origin="lower"); plt.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

    H, W = u0.shape
    dx = 1.0 / (H - 1)
    dy = 1.0 / (W - 1)

    du_dx = (u0[2:, 1:-1] - u0[:-2, 1:-1]) / (2 * dx)
    du_dy = (u0[1:-1, 2:] - u0[1:-1, :-2]) / (2 * dy)
    a_c = a[1:-1, 1:-1]

    vx = -a_c * du_dx
    vy = -a_c * du_dy
    speed = torch.sqrt(vx**2 + vy**2)

    axs[2].set_title("|v| (flux/vitesse)")
    im2 = axs[2].imshow(speed, origin="lower"); plt.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

    # Quiver
    step = max(1, H // 16)
    Y, X = torch.meshgrid(torch.arange(1, H-1), torch.arange(1, W-1), indexing="ij")
    Xq = X[::step, ::step].numpy()
    Yq = Y[::step, ::step].numpy()
    vx_q = vx[::step, ::step].numpy()
    vy_q = vy[::step, ::step].numpy()

    plt.figure(figsize=(4, 4))
    plt.title("Flux v = -a ∇u (quiver)")
    plt.imshow(a, origin="lower", alpha=0.6)
    plt.quiver(Xq, Yq, vx_q, vy_q, color="k", angles="xy", scale_units="xy", scale=1.0)
    plt.tight_layout()
    plt.show()

show_darcy_example(x, y, title="Un exemple Darcy (train)")

print('x:', x.shape)
print('y:', y.shape)

## 4. Losses et métriques : que mesurent-elles ?

NeuralOperator fournit des losses adaptées à l’apprentissage d’opérateurs (champs).

### L2 relative (`LpLoss(d=2,p=2)`)
Elle mesure l’erreur d’amplitude globale :
$$
\text{relL2}(\hat u,u)=\frac{\|\hat u-u\|_2}{\|u\|_2}
$$
Intuition : “est-ce que la prédiction a la bonne **échelle** et la bonne **forme globale** ?”

### H1 (`H1Loss(d=2)`)
La norme $H^1$ compare **la fonction** et aussi ses **dérivées** :
$$
\|\hat u-u\|_{H^1}^2 \approx \|\hat u-u\|_2^2 + \|\nabla \hat u-\nabla u\|_2^2
$$
Intuition : “est-ce que la prédiction respecte aussi la **structure fine** (pentes, gradients) ?”

Dans Darcy, les gradients $\nabla u$ sont directement liés au **flux** :
$$
\mathbf{v}=-a\nabla u
$$
Donc une loss $H^1$ peut améliorer la qualité “physique” du champ (gradients plus réalistes).

### Quel choix pour l’entraînement ?
- Entraîner en **L2** : plus simple, souvent plus stable, mais peut être plus “lisse”.
- Entraîner en **H1** : force la structure fine, parfois plus difficile mais plus fidèle (selon les cas).

> Note : si vous voyez un warning du type “H1Loss received unexpected keyword argument 'x'”, c’est lié à la façon dont `Trainer` passe le batch (`x`, `y`) à la loss. Le `x` est simplement ignoré, ce n’est pas bloquant.

---

In [ ]:
try:
    from neuralop import LpLoss, H1Loss
except Exception:
    from neuralop.losses import LpLoss, H1Loss

l2loss = LpLoss(d=2, p=2)
h1loss = H1Loss(d=2)

if config["opt"]["training_loss"] == "l2":
    train_loss = l2loss
elif config["opt"]["training_loss"] == "h1":
    train_loss = h1loss
else:
    raise ValueError('training_loss must be "l2" or "h1"')

eval_losses = {"l2": l2loss, "h1": h1loss}
print("Training loss:", config["opt"]["training_loss"])

# Petit helper pour les ablations (simple à utiliser en TP)
def set_training_loss(name: str):
    """Met à jour config['opt']['training_loss'] et la variable globale train_loss."""
    global train_loss
    name = name.lower()
    if name == "l2":
        train_loss = l2loss
        config["opt"]["training_loss"] = "l2"
    elif name == "h1":
        train_loss = h1loss
        config["opt"]["training_loss"] = "h1"
    else:
        raise ValueError('name must be "l2" or "h1"')
    print("Training loss set to:", config["opt"]["training_loss"])

## 5. Pipeline d’entraînement (Trainer)

On encapsule la logique `Trainer` pour entraîner U-Net puis (T)FNO.

In [ ]:
# Trainer (API varie selon versions)
try:
    from neuralop.training import Trainer as TrainerCls
except Exception:
    from neuralop import Trainer as TrainerCls

def make_optimizer_and_scheduler(model, opt_cfg):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=opt_cfg["learning_rate"],
        weight_decay=opt_cfg["weight_decay"],
    )

    if opt_cfg["scheduler"] == "CosineAnnealingLR":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=opt_cfg["scheduler_T_max"])
    elif opt_cfg["scheduler"] == "StepLR":
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=opt_cfg["step_size"], gamma=opt_cfg["gamma"])
    elif opt_cfg["scheduler"] == "ReduceLROnPlateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=opt_cfg["gamma"], patience=5, mode="min")
    else:
        raise ValueError(f'Unknown scheduler: {opt_cfg["scheduler"]}')
    return optimizer, scheduler

In [ ]:
def train_with_trainer(model, name):
    model = model.to(device)
    optimizer, scheduler = make_optimizer_and_scheduler(model, config["opt"])

    # Construire Trainer (API récente préférée)
    try:
        trainer = TrainerCls(model=model, n_epochs=config["opt"]["n_epochs"], device=device, verbose=True, data_processor=data_processor)
        trainer_api = "new"
    except TypeError:
        trainer = TrainerCls(model, n_epochs=config["opt"]["n_epochs"], device=device, verbose=True)
        trainer_api = "old"

    #print(f"\n=== Training {name} (Trainer API: {trainer_api}) ===")

    if trainer_api == "new":
        trainer.train(
            train_loader=train_loader,
            test_loaders=test_loaders,
            optimizer=optimizer,
            scheduler=scheduler,
            regularizer=False,
            training_loss=train_loss,
            eval_losses=eval_losses,
        )
    else:
        # fallback ancien
        try:
            trainer.train(
                train_loader, test_loaders,
                data_processor,
                model,
                optimizer,
                scheduler,
                regularizer=False,
                training_loss=train_loss,
                eval_losses=eval_losses,
            )
        except TypeError:
            trainer.train(
                train_loader, test_loaders,
                optimizer=optimizer,
                scheduler=scheduler,
                regularizer=False,
                training_loss=train_loss,
                eval_losses=eval_losses,
            )

    return model

In [ ]:
@torch.no_grad()
def get_loader(test_loaders, idx=0):
    if isinstance(test_loaders, dict):
        keys = list(test_loaders.keys())
        return test_loaders[keys[idx]]
    if isinstance(test_loaders, (list, tuple)):
        return test_loaders[idx]
    return test_loaders

In [ ]:
@torch.no_grad()
def show_batch_predictions(model, loader, title=""):
    model.eval()
    batch = next(iter(loader))

    if isinstance(batch, (list, tuple)):
        xb, yb = batch
        xb = xb.to(device); yb = yb.to(device)
        pred = model(xb)
    else:
        batch = {k: v.to(device) for k, v in batch.items()}
        try:
            pred = model(**batch)
            xb, yb = batch["x"], batch["y"]
        except TypeError:
            xb, yb = batch["x"], batch["y"]
            pred = model(xb)

    x0 = xb[1].detach().cpu()
    y0 = yb[1].detach().cpu()
    p0 = pred[1].detach().cpu()

    a = x0[0]
    u = y0[0] if y0.ndim == 3 else y0.squeeze(0)
    uhat = p0[0] if p0.ndim == 3 else p0.squeeze(0)
    err = uhat - u

    fig, axs = plt.subplots(1, 4, figsize=(14, 3))
    axs[0].set_title("a (input)"); im0 = axs[0].imshow(a, origin="lower"); plt.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)
    axs[1].set_title("u (true)"); im1 = axs[1].imshow(u, origin="lower"); plt.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)
    axs[2].set_title("u_hat (pred)"); im2 = axs[2].imshow(uhat, origin="lower"); plt.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)
    axs[3].set_title("error"); im3 = axs[3].imshow(err, origin="lower"); plt.colorbar(im3, ax=axs[3], fraction=0.046, pad=0.04)

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
@torch.no_grad()
def evaluate_quick(model, loader):
    model.eval()
    l2s, h1s, n = 0.0, 0.0, 0
    for batch in loader:
        if isinstance(batch, (list, tuple)):
            xb, yb = batch
            xb = xb.to(device); yb = yb.to(device)
            pred = model(xb)
        else:
            batch = {k: v.to(device) for k, v in batch.items()}
            try:
                pred = model(**batch)
                yb = batch["y"]
            except TypeError:
                pred = model(batch["x"])
                yb = batch["y"]

        l2s += eval_losses["l2"](pred, yb).item()
        h1s += eval_losses["h1"](pred, yb).item()
        n += 1
    return {"l2": l2s / max(n, 1), "h1": h1s / max(n, 1)}

## 6. Baseline : U‑Net (CNN champ -> champ)

Avant de passer aux opérateurs en Fourier, on veut un **baseline fort** et standard pour des tâches “image->image”.

### Pourquoi U‑Net ?
- C’est une architecture CNN très utilisée (segmentation / débruitage / super‑résolution).
- Elle combine :
  - un **encodeur** (downsampling) qui agrège du contexte,
  - un **décodeur** (upsampling),
  - des **skip connections** qui réinjectent les détails de haute résolution.

### Ce qu’on attend comme comportement
- Très bon sur la **même résolution** que l’entraînement.
- Pas forcément “naturellement” robuste quand on change la résolution (32->64), même si parfois ça marche.

Dans ce TP, l’U‑Net sert à poser la question :
> Est-ce que le (T)FNO apporte quelque chose au‑delà d’un CNN moderne ?

---

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GELU(),
        )

    def forward(self, x):
        return self.net(x)

class Encoder(nn.Module):
    def __init__(self, in_channels, base=16, depth=4):
        super().__init__()
        self.depth = depth

        self.stem = DoubleConv(in_channels, base)
        self.pools = nn.ModuleList()
        self.blocks = nn.ModuleList()

        ch = base
        for _ in range(depth):
            self.pools.append(nn.MaxPool2d(2))
            self.blocks.append(DoubleConv(ch, ch * 2))
            ch *= 2

        self.out_channels = ch
        # canaux des skips : [base, 2base, 4base, ..., 2^depth base]
        self.skip_channels = [base * (2 ** i) for i in range(depth + 1)]

    def forward(self, x):
        skips = []
        x = self.stem(x)
        skips.append(x)
        for i in range(self.depth):
            x = self.pools[i](x)
            x = self.blocks[i](x)
            skips.append(x)
        return x, skips

class Bottleneck(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = DoubleConv(channels, channels)

    def forward(self, x):
        return self.block(x)

class UpBlockT(nn.Module):
    """ConvTranspose2d -> concat skip -> DoubleConv (sans padding)."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = DoubleConv(out_ch + skip_ch, out_ch)

    def forward(self, x_dec, x_skip):
        x_dec = self.up(x_dec)

        # Sans padding: on exige que les tailles soient identiques
        assert x_dec.shape[-2:] == x_skip.shape[-2:], (
            f"Shape mismatch: up={x_dec.shape[-2:]} vs skip={x_skip.shape[-2:]}. "
            "Utilisez des tailles H,W divisibles par 2^depth (ex: 32,64,128) et kernel=2,stride=2."
        )

        x = torch.cat([x_skip, x_dec], dim=1)
        return self.conv(x)

class Decoder(nn.Module):
    def __init__(self, encoder_channels, skip_channels, out_channels=1, depth=4):
        super().__init__()
        self.depth = depth
        ch = encoder_channels

        self.up_blocks = nn.ModuleList()
        for i in range(depth):
            skip_ch = skip_channels[depth - 1 - i]  # skips du bas vers le haut
            out_ch = ch // 2
            self.up_blocks.append(UpBlockT(in_ch=ch, skip_ch=skip_ch, out_ch=out_ch))
            ch = out_ch

        self.head = nn.Conv2d(ch, out_channels, kernel_size=1)

    def forward(self, x, skips):
        for i in range(self.depth):
            skip = skips[self.depth - 1 - i]  # depth-1 ... 0
            x = self.up_blocks[i](x, skip)
        return self.head(x)

class UNetSmall(nn.Module):
    """U-Net Encoder+Bottleneck+Decoder"""
    def __init__(self, in_channels, out_channels=1, base=16, depth=4):
        super().__init__()
        self.encoder = Encoder(in_channels, base=base, depth=depth)
        self.bottleneck = Bottleneck(self.encoder.out_channels)
        self.decoder = Decoder(
            encoder_channels=self.encoder.out_channels,
            skip_channels=self.encoder.skip_channels,
            out_channels=out_channels,
            depth=depth,
        )

    def forward(self, x=None, y=None, **kwargs):
        if x is None and "x" in kwargs:
            x = kwargs["x"]
        x_deep, skips = self.encoder(x)
        x_mid = self.bottleneck(x_deep)
        return self.decoder(x_mid, skips)

## 7. Entraîner la baseline U-Net

Lancez l’entraînement, puis regardez :
- les métriques test (l2, h1)
- les visualisations sur 32×32 et 64×64

In [ ]:
unet = UNetSmall(
    in_channels=in_channels,
    out_channels=1,
    base=config["unet"]["base_channels"],
    depth=config["unet"]["depth"],
)

unet = train_with_trainer(unet, "U-Net")

test32 = get_loader(test_loaders, idx=0)
print("U-Net test32:", evaluate_quick(unet, test32))
show_batch_predictions(unet, test32, title="U-Net — test 32×32")

try:
    test64 = get_loader(test_loaders, idx=1)
    print("U-Net test64:", evaluate_quick(unet, test64))
    show_batch_predictions(unet, test64, title="U-Net — test 64×64 (zero-shot)")
except Exception as e:
    print("Pas de loader 64×64 détecté:", repr(e))

show_batch_predictions(unet, test32, title="U-Net — test 32×32")
show_batch_predictions(unet, test64, title="U-Net — test 64×64 (zero-shot)")

## 8. Modèle principal : (T)FNO (Fourier Neural Operator)

On instancie un **TFNO** (version factorisée) si `factorization` n’est pas `None`, sinon un **FNO** standard.

### Rôle des paramètres (voir la section 0bis)
- `n_modes=(m1,m2)` : nombre de modes gardés en Fourier (basses fréquences).
  - Plus grand -> plus de détails possibles (mais coût ↑).
- `hidden_channels` : largeur interne (capacité du modèle).
- `n_layers` : nombre de blocs empilés (profondeur).

### TFNO vs FNO
- `TFNO` (factorization="tucker") : approxime certains tenseurs de poids pour réduire le coût mémoire/temps.
- `FNO` : plus “direct”, mais peut devenir coûteux si on augmente beaucoup la taille.

On peut commencer avec :
- `hidden_channels` petit (8–32),
- `n_modes` modéré (8–16),
- `n_epochs` petit (5–20).

---

In [ ]:
# Import modèles (les noms exacts varient selon versions)
try:
    from neuralop.models import FNO as FNO2d
    from neuralop.models import TFNO as TFNO2d
except Exception:
    from neuralop.models import FNO2d as FNO2d
    from neuralop.models import TFNO2d as TFNO2d

## 9. Entraîner (T)FNO + évaluer (32×32, 64×64)

Comparez directement aux résultats U-Net.

In [ ]:
def make_fno(kind="tfno", n_modes=None, hidden_channels=None, n_layers=None):
    """Construit un FNO ou un TFNO avec la config courante.

    kind = "fno"  -> FNO standard (poids spectraux pleins)
           "tfno" -> version factorisee (Tucker), moins de parametres
    """
    cls = TFNO2d if kind == "tfno" else FNO2d
    kwargs = dict(
        n_modes=n_modes or config["fno"]["n_modes"],
        hidden_channels=hidden_channels or config["fno"]["hidden_channels"],
        in_channels=in_channels,
        out_channels=1,
        n_layers=n_layers or config["fno"]["n_layers"],
    )
    return cls(**kwargs)

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# --- Entrainement du FNO ---
fno = make_fno("fno")
print("FNO  :", count_params(fno), "parametres")
fno = train_with_trainer(fno, name="FNO")

# --- Entrainement du TFNO ---
# ATTENTION au bug que j'avais laisse ici : j'ecrivais
#     Tfno = train_with_trainer(fno, name="fFNO")
# ce qui re-entrainait le FNO et rangeait le resultat dans Tfno. Les deux
# variables pointaient alors sur le MEME modele, et toute la comparaison
# FNO / TFNO qui suivait ne comparait rien du tout.
Tfno = make_fno("tfno")
print("TFNO :", count_params(Tfno), "parametres")
Tfno = train_with_trainer(Tfno, name="TFNO")

test32 = get_loader(test_loaders, idx=0)
test64 = get_loader(test_loaders, idx=1)

print("\n--- Resultats ---")
print("U-Net test32:", evaluate_quick(unet, test32))
print("U-Net test64:", evaluate_quick(unet, test64))
print("FNO   test32:", evaluate_quick(fno, test32))
print("FNO   test64:", evaluate_quick(fno, test64))
print("TFNO  test32:", evaluate_quick(Tfno, test32))
print("TFNO  test64:", evaluate_quick(Tfno, test64))

show_batch_predictions(fno, test32, title="FNO — test 32×32")
show_batch_predictions(fno, test64, title="FNO — test 64×64 (zero-shot)")
show_batch_predictions(Tfno, test64, title="TFNO — test 64×64 (zero-shot)")
show_batch_predictions(unet, test64, title="U-Net — test 64×64 (zero-shot)")

**Le resultat central du TP est dans ces quatre figures.**

En 32x32 (la resolution d'entrainement), U-Net et FNO se valent a peu pres.
En 64x64 **sans reentrainement**, l'ecart est spectaculaire : le FNO produit
encore un champ correct, l'U-Net s'effondre.

Pourquoi ? Parce que les deux ne parametrent pas la meme chose.

- Une **convolution** apprend des poids attaches a une grille de pixels. Un
  filtre 3x3 couvre 3/32 de l'image en 32x32, mais seulement 3/64 en 64x64 :
  le meme filtre ne "voit" plus la meme echelle physique. Le reseau change donc
  de comportement quand la grille change.
- Une **couche de Fourier** apprend des poids attaches a des **modes**
  (frequences), qui sont definis sur le domaine continu et non sur la grille.
  Le mode k=3 est le mode k=3, que l'on echantillonne le domaine avec 32 ou
  64 points. C'est ce qui rend le FNO *discretization invariant* : il approxime
  un operateur entre espaces de fonctions, pas une application entre tableaux
  de pixels.

## 10. Ablations : protocole + interprétation

Ici, l'objectif n'est pas de "faire du tuning" à l'infini, mais de **comprendre** ce que contrôlent les hyperparamètres.

- Choisissez un "réglage de base" (celui du notebook).
- **Changez un seul paramètre à la fois**, gardez le reste identique.
- Pour CPU : mettez `n_train=128–256` et `n_epochs=5–10` pour que ça tourne vite.
- Pour chaque run, notez :
  - `L2` et `H1` sur test 32×32
  - `L2` et `H1` sur test 64×64
  - 1 ou 2 figures qualitatives

### Ablations conseillées
1) **Loss** : `training_loss="l2"` vs `"h1"`
2) **Modes** : `n_modes=(8,8)`, `(12,12)`, `(16,16)`
3) **Capacité** : `hidden_channels=8, 16, 32`
4) (Option GPU) **TFNO vs FNO**

### Questions d'interprétation
1. Sur quels aspects U-Net et (T)FNO se trompent-ils "différemment" ?
2. Pourquoi `n_modes` contrôle fortement le niveau de détail ?
3. Que signifie "zero-shot super-resolution" dans ce contexte ?

### Bonus
Calculez et comparez $\|\mathbf{v}\|$ avec $\mathbf{v}=-a\nabla u$ pour la vérité et la prédiction $\hat u$.

---

**ATTENTION - le bug qui invalidait toutes mes ablations.**
J'avais ecrit :

```python
config["opt"]["training_loss"] == "l2"     # <- DEUX signes egal !
```

`==` est un test d'egalite : cette ligne calcule un booleen, ne l'utilise pas,
et ne modifie **rien**. Tous mes runs "l2" et "h1" utilisaient donc la meme
loss (celle fixee au depart), et les differences observees n'etaient que du
bruit d'initialisation.

Et meme avec un seul `=`, cela n'aurait pas suffi : `train_with_trainer` lit la
variable **globale** `train_loss`, pas le dictionnaire `config`. C'est
exactement pour cela que la fonction `set_training_loss()` existe plus haut
dans le notebook. C'est un bon exemple d'un piege classique : une variable
"config" qui n'est lue qu'a un seul endroit, au moment de l'initialisation.

In [ ]:
resultats = []

In [ ]:
def ablation(nom, model, loss_name):
    """Entraine un modele avec une loss donnee et enregistre les metriques."""
    set_training_loss(loss_name)            # <- met bien a jour la globale train_loss
    m = train_with_trainer(model, name=nom)
    r32 = evaluate_quick(m, test32)
    r64 = evaluate_quick(m, test64)
    ligne = dict(modele=nom, loss=loss_name, params=count_params(m),
                 l2_32=r32["l2"], h1_32=r32["h1"], l2_64=r64["l2"], h1_64=r64["h1"])
    resultats.append(ligne)
    print(f"{nom:22s} | loss {loss_name} | test32 L2 {r32['l2']:.4f} H1 {r32['h1']:.4f}"
          f" | test64 L2 {r64['l2']:.4f} H1 {r64['h1']:.4f}")
    return m

### Ablation 1 : loss L2 vs H1

In [ ]:
for loss_name in ("l2", "h1"):
    ablation(f"FNO ({loss_name})", make_fno("fno"), loss_name)
    ablation(f"TFNO ({loss_name})", make_fno("tfno"), loss_name)
    ablation(f"U-Net ({loss_name})",
             UNetSmall(in_channels=in_channels, out_channels=1,
                       base=config["unet"]["base_channels"],
                       depth=config["unet"]["depth"]),
             loss_name)

### Ablation 2 : nombre de modes de Fourier

In [ ]:
set_training_loss("h1")
modeles_modes = {}
for m1 in (4, 8, 12, 16, 24):
    modeles_modes[m1] = ablation(f"FNO modes ({m1},{m1})",
                                 make_fno("fno", n_modes=(m1, m1)), "h1")

### Ablation 3 : capacite (hidden_channels)

In [ ]:
for hc in (8, 16, 32):
    ablation(f"FNO hidden {hc}", make_fno("fno", hidden_channels=hc), "h1")

### Tableau recapitulatif

In [ ]:
import pandas as pd
df = pd.DataFrame(resultats)
print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# Ecart de generalisation en resolution : combien perd-on en passant a 64x64 ?
df["degradation_32_64"] = df["l2_64"] / df["l2_32"]
print("\nRapport L2(64) / L2(32)  (1.0 = aucune degradation) :")
print(df[["modele", "loss", "degradation_32_64"]].to_string(index=False,
                                                           float_format=lambda v: f"{v:.2f}"))

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
ms = sorted(modeles_modes.keys())
l32 = [next(r["l2_32"] for r in resultats if r["modele"] == f"FNO modes ({m},{m})") for m in ms]
l64 = [next(r["l2_64"] for r in resultats if r["modele"] == f"FNO modes ({m},{m})") for m in ms]
plt.plot(ms, l32, 'o-', label="test 32x32")
plt.plot(ms, l64, 's-', label="test 64x64")
plt.xlabel("nombre de modes gardes"); plt.ylabel("erreur L2 relative")
plt.title("Effet du nombre de modes"); plt.legend(); plt.grid(alpha=.3)

plt.subplot(1, 2, 2)
sous = df[df.modele.str.contains("FNO hidden")]
plt.plot([8, 16, 32], sous["l2_32"], 'o-', label="test 32x32")
plt.plot([8, 16, 32], sous["l2_64"], 's-', label="test 64x64")
plt.xlabel("hidden_channels"); plt.ylabel("erreur L2 relative")
plt.title("Effet de la capacite"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

### Reponses aux questions d'interpretation

**1. U-Net et FNO se trompent-ils differemment ?**

Oui, et de facon tres caracteristique.

- Le **FNO** tronque les hautes frequences par construction : au-dela du mode
  `n_modes`, l'information est purement et simplement supprimee. Ses erreurs
  sont donc du **lissage** : les fronts raides et les discontinuites de
  permeabilite sont arrondis. L'erreur est repartie de facon assez uniforme sur
  le domaine, et ressemble a un residu basse frequence.
- L'**U-Net**, lui, est local. Il reproduit bien les details fins la ou il a vu
  des configurations similaires, mais peut se tromper sur le **niveau global**
  du champ (un biais constant sur une region entiere), parce qu'aucun mecanisme
  ne lui garantit une coherence a longue distance. Ses erreurs sont plus
  "par plaques", parfois avec des artefacts en damier aux bords des blocs
  d'upsampling (`ConvTranspose2d`).

En changeant de resolution, l'U-Net produit en plus des artefacts francs,
puisque ses filtres ne correspondent plus a la meme echelle physique.

**2. Pourquoi `n_modes` controle-t-il le niveau de detail ?**

Parce que c'est litteralement une **troncature spectrale**. La couche de Fourier
calcule la FFT, ne garde que les `n_modes` premiers coefficients, leur applique
une transformation lineaire apprise, puis fait la FFT inverse. Tout ce qui est
au-dela de `n_modes` est mis a zero : le modele est **structurellement
incapable** de produire des variations plus fines que la longueur d'onde
correspondante.

C'est le theoreme d'echantillonnage vu sous un autre angle : garder $m$ modes
revient a limiter la resolution effective a environ $2m$ points par dimension.
D'ou le comportement observe : trop peu de modes -> sortie floue ; beaucoup de
modes -> plus de details, mais un cout en $O(m^2)$ en parametres et un risque
de surapprentissage sur les hautes frequences, souvent bruitees.

**3. "Zero-shot super-resolution" : est-ce vraiment de la super-resolution ?**

Non, pas au sens du traitement d'image, et c'est un abus de langage important a
relever.

En imagerie, la super-resolution consiste a **inventer** de l'information
absente d'une image basse resolution. Ici, ce n'est pas ce qui se passe : on
recoit une **entree** $a$ deja echantillonnee en 64x64 (donc plus riche), et on
demande au modele de produire la sortie sur cette meme grille fine. On
n'invente rien - on evalue un operateur appris sur une discretisation plus fine
des memes fonctions.

La formulation correcte est **invariance a la discretisation** : le FNO
approxime un operateur $\mathcal{G}: a(\cdot) \mapsto u(\cdot)$ entre espaces de
fonctions, et la grille n'est qu'un moyen de representer ces fonctions. C'est
d'ailleurs pour cela que ca marche : l'operateur appris ne depend pas de la
grille, donc changer de grille est licite.

### Bonus : verification "physique" via le flux

Le flux $\mathbf{v} = -a\nabla u$ depend des **gradients** de $u$. C'est donc un
test bien plus severe qu'une erreur sur $u$ : deux champs peuvent etre proches
en norme $L^2$ et avoir des gradients tres differents. C'est exactement ce que
la loss $H^1$ cherche a controler.

In [ ]:
@torch.no_grad()
def comparer_flux(model, loader, idx=1):
    """Compare le flux v = -a grad(u) entre verite et prediction."""
    model.eval()
    batch = next(iter(loader))
    if isinstance(batch, (list, tuple)):
        xb, yb = batch
    else:
        xb, yb = batch["x"], batch["y"]
    xb, yb = xb.to(device), yb.to(device)
    pred = model(xb)

    a = xb[idx, 0].cpu()
    u = yb[idx, 0].cpu()
    uh = pred[idx, 0].cpu()

    def flux(u_field, a_field):
        H, W = u_field.shape
        dx, dy = 1.0 / (H - 1), 1.0 / (W - 1)
        du_dx = (u_field[2:, 1:-1] - u_field[:-2, 1:-1]) / (2 * dx)
        du_dy = (u_field[1:-1, 2:] - u_field[1:-1, :-2]) / (2 * dy)
        ac = a_field[1:-1, 1:-1]
        vx, vy = -ac * du_dx, -ac * du_dy
        return torch.sqrt(vx ** 2 + vy ** 2)

    v_vrai, v_pred = flux(u, a), flux(uh, a)
    err_u = (uh - u).norm() / u.norm()
    err_v = (v_pred - v_vrai).norm() / v_vrai.norm()

    fig, axs = plt.subplots(1, 4, figsize=(15, 3.2))
    for ax, (img, t) in zip(axs, [(u, "u vrai"), (uh, "u predit"),
                                  (v_vrai, "|v| vrai"), (v_pred, "|v| predit")]):
        im = ax.imshow(img, origin="lower"); ax.set_title(t)
        plt.colorbar(im, ax=ax, fraction=.046)
    plt.suptitle(f"erreur relative sur u : {err_u:.4f}   |   sur le flux |v| : {err_v:.4f}")
    plt.tight_layout(); plt.show()
    return err_u.item(), err_v.item()

In [ ]:
print("=== FNO entraine en L2 ===")
set_training_loss("l2")
fno_l2 = train_with_trainer(make_fno("fno"), name="FNO l2")
eu_l2, ev_l2 = comparer_flux(fno_l2, test32)

print("=== FNO entraine en H1 ===")
set_training_loss("h1")
fno_h1 = train_with_trainer(make_fno("fno"), name="FNO h1")
eu_h1, ev_h1 = comparer_flux(fno_h1, test32)

print(f"\n{'entrainement':16s} {'erreur sur u':>14s} {'erreur sur le flux':>20s}")
print(f"{'loss L2':16s} {eu_l2:14.4f} {ev_l2:20.4f}")
print(f"{'loss H1':16s} {eu_h1:14.4f} {ev_h1:20.4f}")

**C'est la justification de la loss $H^1$.** Entrainer en $L^2$ donne une
erreur legerement plus faible... **sur $u$**, ce qui est logique puisque c'est
exactement ce qu'on a optimise. Mais sur le **flux** - la quantite qui a un sens
physique, celle qu'un ingenieur va reellement utiliser - le modele entraine en
$H^1$ est meilleur.

Lecon generale : **la loss encode ce qui compte pour vous.** Optimiser une
metrique commode n'est pas la meme chose qu'optimiser la bonne. C'est le meme
raisonnement qui menait, au TP RNN, a preferer un entrainement multi-pas quand
l'objectif reel est la prevision long terme.

## Bonus: Aller plus loin en mixant dépendance spatiale et temporelle

Dans Darcy Flow, on apprend un opérateur **statique** : $a(x,y)\mapsto u(x,y)$.
Mais beaucoup de systèmes physiques sont **spatio-temporels** : on observe un champ $u_t(x,y)$ qui évolue dans le temps.

Un exemple classique est **Navier–Stokes 2D** (écoulement incompressible), où l'état du fluide (souvent la vorticité ou la vitesse) suit une dynamique :
$$
u_{t+1} = \mathcal{F}(u_t)
$$

### Rollout temporel (analogue direct du TP RNN)
Si un modèle prédit $u_{t+1}$ à partir de $u_t$, on peut faire un **rollout autoregressif** :
$$
\hat u_{t+1}=\mathcal{F}(\hat u_t),\ \hat u_{t+2}=\mathcal{F}(\hat u_{t+1}),\dots
$$
Comme pour les RNN, les erreurs peuvent **s'accumuler** au fil du temps (drift).

### Objectif du bonus
- Charger un petit dataset Navier–Stokes
- Récupérer une séquence $u_0,\dots,u_T$
- Faire un rollout autoregressif
- Tracer l'erreur en fonction du temps

In [ ]:
from neuralop.data.datasets import NavierStokesDataset

### Bonus : Rollout temporel sur Navier-Stokes 2D

Nous utilisons un dataset de vorticité. Le modèle prend la frame à l'instant $t$ et prédit l'instant $t+1$. En mode test, on réinjecte la prédiction pour prédire $t+2$, et ainsi de suite.

C'est **exactement** le probleme du TP meteo, transpose en 2D spatial. Le
modele est entraine en "teacher forcing" (une seule etape, avec la vraie
frame en entree) mais utilise en boucle fermee. Les memes remedes s'appliquent :
entrainement multi-pas, ou scheduled sampling.

In [ ]:
def charger_navier_stokes(n_train=200, n_test=20, resolution=64, T=20):
    """Charge un jeu Navier-Stokes et le met en forme (N, T, H, W).

    L'API de NeuralOperator change souvent entre versions ; on encapsule le
    chargement pour n'avoir a corriger qu'a un seul endroit.
    """
    ds = NavierStokesDataset(
        root_dir="./data", n_train=n_train, n_tests=[n_test],
        batch_size=8, test_batch_sizes=[8],
        train_resolution=resolution, test_resolutions=[resolution],
    )
    return ds

In [ ]:
def entrainer_pas_de_temps(model, sequences, n_epochs=20, lr=1e-3, bsz=8):
    """Entraine un FNO a predire u_{t+1} a partir de u_t.

    sequences : tenseur (N, T, H, W) de trajectoires.
    On fabrique des paires (u_t, u_{t+1}) en aplatissant les dimensions N et T.
    """
    N, T, H, W = sequences.shape
    X = sequences[:, :-1].reshape(-1, 1, H, W)     # toutes les frames sauf la derniere
    Y = sequences[:, 1:].reshape(-1, 1, H, W)      # decalees d'un pas
    X, Y = X.to(device), Y.to(device)

    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = LpLoss(d=2, p=2)

    for epoch in range(n_epochs):
        model.train()
        perm = torch.randperm(len(X), device=device)
        tot = 0.0
        for k in range(0, len(X), bsz):
            idx = perm[k:k + bsz]
            loss = crit(model(X[idx]), Y[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        if epoch % max(1, n_epochs // 5) == 0:
            print(f"epoch {epoch:3d} | loss {tot/len(X):.5f}")
    return model

In [ ]:
@torch.no_grad()
def rollout(model, u0, n_steps):
    """Rollout autoregressif : on part de u0 et on itere le modele sur lui-meme."""
    model.eval()
    u = u0.to(device)
    trajectoire = []
    for _ in range(n_steps):
        u = model(u)                # la sortie devient l'entree suivante
        trajectoire.append(u.cpu())
    return torch.cat(trajectoire, dim=0)

### Execution du bonus

Le telechargement du dataset Navier-Stokes est lourd (~1 Go) et l'entrainement
demande un GPU. On protege donc l'execution.

In [ ]:
FAIRE_LE_BONUS = torch.cuda.is_available()

if FAIRE_LE_BONUS:
    try:
        ns = charger_navier_stokes()
        # Mise en forme : on recupere un tenseur (N, T, H, W) de trajectoires.
        # Le nom exact des attributs depend de la version de neuraloperator ;
        # on inspecte le premier batch pour s'adapter.
        batch = next(iter(ns.train_db)) if hasattr(ns, "train_db") else None
        print("structure du dataset :", type(batch), getattr(batch, "keys", lambda: None)())

        # A adapter selon la sortie ci-dessus : on suppose ici des trajectoires
        # stockees dans un tenseur de shape (N, T, H, W).
        sequences = ns.train_db.data if hasattr(ns.train_db, "data") else None

        if sequences is not None and sequences.ndim == 4:
            model_ns = FNO2d(n_modes=(16, 16), hidden_channels=32,
                             in_channels=1, out_channels=1, n_layers=4)
            model_ns = entrainer_pas_de_temps(model_ns, sequences[:180], n_epochs=20)

            # Rollout sur une trajectoire de test
            traj_vraie = sequences[-1]                       # (T, H, W)
            u0 = traj_vraie[0:1].unsqueeze(1)                # (1, 1, H, W)
            n_steps = traj_vraie.shape[0] - 1
            traj_pred = rollout(model_ns, u0, n_steps)[:, 0] # (n_steps, H, W)

            # Erreur en fonction du temps
            erreurs = [((traj_pred[t] - traj_vraie[t + 1]).norm()
                        / traj_vraie[t + 1].norm()).item() for t in range(n_steps)]

            plt.figure(figsize=(7, 4))
            plt.plot(range(1, n_steps + 1), erreurs, 'o-')
            plt.xlabel("pas de temps"); plt.ylabel("erreur L2 relative")
            plt.title("Accumulation de l'erreur en rollout autoregressif")
            plt.grid(alpha=.3); plt.show()

            # Visualisation de la derive
            pas = [0, n_steps // 3, 2 * n_steps // 3, n_steps - 1]
            fig, axs = plt.subplots(2, len(pas), figsize=(3.5 * len(pas), 7))
            for j, t in enumerate(pas):
                axs[0, j].imshow(traj_vraie[t + 1], origin="lower")
                axs[0, j].set_title(f"verite, t = {t+1}")
                axs[1, j].imshow(traj_pred[t], origin="lower")
                axs[1, j].set_title(f"predit, t = {t+1} (err {erreurs[t]:.3f})")
                for a in (axs[0, j], axs[1, j]): a.axis('off')
            plt.suptitle("Derive progressive du rollout")
            plt.tight_layout(); plt.show()
        else:
            print("Format de dataset inattendu : adapter l'extraction des trajectoires.")
    except Exception as e:
        print("Bonus non execute :", repr(e))
else:
    print("Bonus Navier-Stokes ignore (pas de GPU).")
    print("Comportement attendu : l'erreur croit d'abord lentement puis")
    print("explose - la meme accumulation d'erreurs qu'au TP RNN meteo.")

## Conclusion du TP

| | U-Net (CNN) | FNO |
|---|---|---|
| Objet appris | application grille -> grille | operateur fonction -> fonction |
| Localite | filtres locaux (3x3) | global par nature (la FFT melange tout le domaine) |
| Changement de resolution | il faut reentrainer | fonctionne tel quel |
| Cout d'une couche | $O(N)$ en pixels | $O(N \log N)$ (FFT) |
| Limite | champ receptif fini | troncature des hautes frequences |

### Le fil rouge de tout le cours

Ce TP boucle la boucle sur quelque chose qui traverse les six precedents : **le
bon modele est celui dont la structure encode les bonnes invariances du
probleme.**

| Structure du probleme | Architecture adaptee | Invariance encodee |
|---|---|---|
| Aucune (vecteurs quelconques) | MLP | aucune |
| Voisinage local, motifs repetes | CNN | translation |
| Sequence, ordre temporel | RNN / GRU / LSTM | translation dans le temps |
| Relations a longue portee, ordre souple | Transformer | permutation (+ positions ajoutees) |
| Fonction sur un domaine continu | FNO | **discretisation** |

Un MLP suffisamment gros pourrait en theorie tout apprendre (theoreme
d'approximation universelle). En pratique il faudrait des quantites de donnees
astronomiques, parce qu'il devrait *deduire* de ces donnees des structures que
les autres architectures lui donnent **gratuitement**. Choisir une architecture,
c'est choisir ce qu'on n'aura pas besoin d'apprendre.